In [2]:
pip install beautifulsoup4 cloudscraper requests selenium

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from deep_translator import GoogleTranslator

input_file = "ao-liem.txt"
output_file = "output_es.txt"

def split_text(text, max_len=5000):
    parts = []
    while len(text) > max_len:
        cut = text.rfind(".", 0, max_len)
        if cut == -1:
            cut = max_len
        parts.append(text[:cut+1].strip())
        text = text[cut+1:]
    if text.strip():  # chỉ thêm nếu không rỗng
        parts.append(text.strip())
    return parts

# Đọc file gốc
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Chia nhỏ và dịch
translated_parts = []
for part in split_text(text):
    try:
        translated = GoogleTranslator(source="auto", target="es").translate(part)
        translated_parts.append(translated)
    except Exception as e:
        print("❌ Lỗi khi dịch đoạn:", e)
        translated_parts.append(part)  # nếu lỗi thì giữ nguyên

# Ghi ra file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(translated_parts))

print("✅ Đã dịch xong! File lưu tại:", output_file)


✅ Đã dịch xong! File lưu tại: output_es.txt


# **TocTruyen**

In [3]:
"""
Crawler cho toctruyen.net — bản tối ưu.

Thay đổi so với bản Selenium cũ
--------------------------------
1. Bỏ hẳn Selenium/Chrome. Danh sách chương lấy qua JSON API nội bộ của site
   (`GET /content/subitems?pid=...`), còn nội dung chương nằm sẵn trong HTML tĩnh.
   -> không tốn 3-5s sleep/chương, không tốn ~2-3s khởi động trình duyệt.
2. API trả TOÀN BỘ chương. Scroll DOM chỉ thấy ~10 chương đầu (nên bản cũ
   `end_chap=6` an toàn chứ 50 chương sẽ thiếu), và phát hiện chương bằng cách
   đọc số trong `title`, dễ sai với "Chương 1.5", "Ngoại truyện".
3. Một `requests.Session` + connection pool keep-alive: reused TLS connection
   cho mọi chương, thay vì handshake lại từ đầu.
4. Tải chương song song bằng ThreadPoolExecutor (I/O-bound), giữ nguyên thứ tự
   chương khi ghi file.
5. Regex biên dịch 1 lần ở cấp module; parser `lxml` (dung lỗi HTML tốt hơn
   `html.parser`; về tốc độ hai cái gần như ngang nhau ở quy mô 1 chương).
6. Bỏ heuristic gộp câu theo dấu câu (nguyên nhân làm mất/hợp nhất đoạn văn):
   site đã đánh dấu mỗi đoạn bằng một <p>.

Chỉ đổi cách lấy & tốc độ; output vẫn là text thuần, các chương cách nhau 1 dòng trống.
"""

from __future__ import annotations

import os
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter

try:
    import certifi
except ImportError:  # certifi tuỳ chọn, có hệ thống CA mặc định để thay
    certifi = None

# ============================ Cấu hình ============================

BASE_URL = "https://toctruyen.net"
UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36"
)
MAX_WORKERS = 4          # tải song song; giảm nếu site trả 429/503
REQUEST_TIMEOUT = 20
DELAY_BETWEEN_TRIES = 2  # nghỉ giữa các lần retry

CHAP_NUM_RE = re.compile(r"chuong-(\d+)", re.I)
ANY_NUM_RE = re.compile(r"(\d+)")
PAGE_NUM_RE = re.compile(r"^\d{1,3}$")           # <p>1</p>, <p>2</p> lạc vào nội dung
# Chỉ bỏ dòng kết truyện. "……" / "…" KHÔNG nằm ở đây: trong truyện nó là nhịp im lặng
# của nhân vật, là nội dung thật.
END_MARKERS = {"hết.", "hết", "hết chương", "hoàn", "hoàn thành", "(hoàn thành)", "full"}
MULTISPACE_RE = re.compile(r"[ \t ]+")
LEAD_PUNCT_RE = re.compile(r"^\s+([,.!?;:])")

# ============================ HTTP layer ============================


def find_ca_bundle() -> str | bool:
    """Đường dẫn CA bundle dùng được, hoặc True để requests tự tìm.

    `certifi` theo pip thường hỏng khi venv bị đổi tên/di chuyển (venv có đường dẫn
    absolute hard-code trong các file *.pth và sitecustomize), gây
    `OSError: Could not find a suitable TLS CA certificate bundle`. Khi đó rơi về
    bundle của hệ thống macOS thay vì ném lỗi.
    """
    candidates = []
    if certifi is not None:
        try:
            candidates.append(certifi.where())
        except Exception:
            pass
    candidates += ["/etc/ssl/cert.pem", "/etc/pki/tls/certs/ca-bundle.crt",
                   "/usr/share/ca-certificates/mozilla/provider.crt"]
    for path in candidates:
        try:
            if path and os.path.isfile(path) and os.path.getsize(path) > 1024:
                return path
        except OSError:
            continue
    return True


def make_session(cookies: dict | None = None) -> requests.Session:
    """Session dùng keep-alive + retry có backoff cho lỗi mạng/5xx thoáng qua."""
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": UA,
            "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
            "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
        }
    )
    session.verify = find_ca_bundle()
    if cookies:
        session.cookies.update(cookies)

    adapter = HTTPAdapter(
        pool_connections=MAX_WORKERS + 2,
        pool_maxsize=MAX_WORKERS + 2,
        max_retries=0,  # chủ động retry trong _get() để tự kiểm soát backoff
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def _get(session: requests.Session, url: str, **kw) -> requests.Response | None:
    """GET với retry + backoff; trả None nếu vẫn thất bại."""
    for attempt in range(3):
        try:
            res = session.get(url, timeout=REQUEST_TIMEOUT, **kw)
            if res.status_code == 200:
                res.encoding = res.encoding or "utf-8"
                return res
            if res.status_code in (429, 500, 502, 503, 504):
                time.sleep(DELAY_BETWEEN_TRIES * (attempt + 1) + random.random())
                continue
            return None
        except (requests.RequestException, OSError) as e:
            # OSError bắt các lỗi cấu hình transport (thiếu CA bundle, SSL context)
            if isinstance(e, OSError) and not isinstance(e, requests.RequestException):
                session.verify = True  # để urllib3 tự lấy CA của hệ thống
            time.sleep(DELAY_BETWEEN_TRIES * (attempt + 1))
    return None


# ============================ Parse URL ============================


def parse_story_url(truyen_url: str) -> tuple[str, str, str]:
    """'https://toctruyen.net/truyen/avlBQwkA/ban-trai-...' -> (pid, slug, href)."""
    path = truyen_url.split("#")[0].split("?")[0].strip("/").rstrip("/")
    parts = path.split("/")
    slug = parts[-1]
    pid = parts[-2] if len(parts) >= 2 else ""
    return pid, slug, truyen_url


# ============================ Danh sách chương ============================


def get_chapter_list(session: requests.Session, truyen_url: str) -> list[tuple[int, str]]:
    """Lấy toàn bộ chương qua JSON API của site (nhanh hơn và đủ hơn scroll DOM).

    API là cách site tự tải danh sách chương, nên trả đủ mọi chương; scroll DOM chỉ
    thấy ~10 chương đầu. Nếu API đổi schema/hỏng thì fallback về parsing HTML.

    Trả về [(số_chương, url), ...] sort giảm dần theo số chương; hàm crawl sort lại
    tăng dần khi ghi file.
    """
    pid, slug, story_href = parse_story_url(truyen_url)
    headers = {"Referer": story_href, "X-Requested-With": "XMLHttpRequest",
               "Accept": "application/json, text/javascript, */*; q=0.01"}

    chapters: list[tuple[int, str]] = []
    seen: set[str] = set()

    res = _get(session, f"{BASE_URL}/content/subitems", params={"pid": pid}, headers=headers)
    if res is not None:
        try:
            payload = res.json()
        except ValueError:
            payload = None
        if payload and payload.get("error") in (0, None):
            for item in (payload.get("data") or {}).get("e") or []:
                # item = [id, type, title, url, date, views, locked, ...]
                if len(item) < 4:
                    continue
                title, link = item[2], item[3]
                if not link or link in seen:
                    continue
                num = _chapter_number(link, title)
                if num is None:
                    continue
                seen.add(link)
                chapters.append((num, urljoin(BASE_URL, link)))

    if not chapters:
        # API hỏng/đổi schema -> đọc anchor ngay trên trang truyện (chỉ ~10 chương đầu)
        print("⚠️ API danh sách chương không dùng được, fallback về parsing HTML "
              "(chỉ lấy được các chương render sẵn trong trang).")
        page = _get(session, story_href)
        if page is not None:
            soup = BeautifulSoup(page.text, "lxml")
            for a in soup.select("div.subitem-item div.name a[href]"):
                link = a["href"]
                if link in seen:
                    continue
                num = _chapter_number(link, a.get("title") or a.get_text(strip=True))
                if num is None:
                    continue
                seen.add(link)
                chapters.append((num, urljoin(BASE_URL, link)))

    if not chapters:
        raise RuntimeError("Không lấy được danh sách chương — kiểm tra lại URL truyện.")

    chapters.sort(key=lambda x: x[0], reverse=True)
    return chapters


def _chapter_number(link: str, title: str) -> int | None:
    """Ưu tiên '...-chuong-12' trong URL, fallback vào số đầu tiên trong title."""
    m = CHAP_NUM_RE.search(link or "")
    if m:
        return int(m.group(1))
    m = ANY_NUM_RE.search(title or "")
    return int(m.group(1)) if m else None


# ============================ Nội dung chương ============================


CONTENT_SELECTORS = (
    "section.container.novel-reading-content",
    "#chapter-content, .chapter-content, #content-full",
)

def _parse(html: str, parser: str) -> BeautifulSoup | None:
    """Parse với parser chỉ định; trả None nếu parser không có sẵn (vd thiếu lxml)."""
    try:
        return BeautifulSoup(html, parser)
    except Exception:
        return None


def extract_paragraphs(html: str) -> list[str]:
    """Lấy các đoạn văn từ khối nội dung, bỏ script/style/rác đánh số trang."""
    block = None
    for parser in ("lxml", "html.parser"):
        soup = _parse(html, parser)
        if soup is None:
            continue
        for sel in CONTENT_SELECTORS:
            block = soup.select_one(sel)
            if block is not None:
                break
        if block is not None:
            break
    if block is None:
        return []

    # Class của khối chứa text là ngẫu nhiên theo từng truyện (vd: div.t4a458058),
    # nên chọn con trực tiếp có nhiều đoạn nhất thay vì hard-code tên class.
    candidates = [d for d in block.find_all("div", recursive=False)]
    if candidates:
        block = max(candidates, key=lambda d: len(d.find_all("p")))

    for tag in block(["script", "style", "ins", "iframe", "center", "h1", "h2", "h3"]):
        tag.decompose()

    paragraphs: list[str] = []
    for p in block.find_all("p"):
        text = MULTISPACE_RE.sub(" ", p.get_text(" ", strip=True)).strip()
        text = LEAD_PUNCT_RE.sub(r"\1", text)
        if not text or PAGE_NUM_RE.fullmatch(text):
            continue
        if text.lower() in END_MARKERS:
            continue
        paragraphs.append(text)

    # truyện có thể dùng <br> thay vì <p>
    if not paragraphs:
        for line in block.get_text("\n", strip=True).splitlines():
            line = MULTISPACE_RE.sub(" ", line).strip()
            if line and not PAGE_NUM_RE.fullmatch(line) and line.lower() not in END_MARKERS:
                paragraphs.append(line)

    return paragraphs


def download_chapter(session: requests.Session, chap_num: int, link: str) -> tuple[int, str]:
    """Tải 1 chương -> (số_chương, text). Lỗi thì trả text báo lỗi, không ném exception
    để các chương còn lại vẫn ghi được."""
    res = _get(session, link)
    if res is None:
        return chap_num, f"Chương {chap_num}\n⚠️ Không tải được trang"
    paragraphs = extract_paragraphs(res.text)
    if not paragraphs:
        return chap_num, f"Chương {chap_num}\n⚠️ Không lấy được nội dung"
    return chap_num, "\n\n".join(paragraphs)


# ============================ Crawl chính ============================


def crawl_toctruyen(
    truyen_url: str,
    start_chap: int,
    end_chap: int,
    output_file: str | None = None,
    *,
    workers: int = MAX_WORKERS,
    delay: float = 0.15,
    session: requests.Session | None = None,
) -> str:
    """Cào chương [start_chap, end_chap] của truyện, ghi ra file, trả về đường dẫn.

    delay: nghỉ ngẫu nhiên nhỏ giữa các request để lịch sự với server.
    """
    session = session or make_session()
    t0 = time.perf_counter()

    chapters = get_chapter_list(session, truyen_url)
    total = chapters[0][0] if chapters else 0
    print(f"📚 Truyện có {len(chapters)} chương (cao nhất: chương {total})")

    selected = [(n, link) for n, link in chapters if start_chap <= n <= end_chap]
    selected.sort(key=lambda x: x[0])
    if not selected:
        print(f"⚠️ Không có chương nào trong khoảng {start_chap}→{end_chap}")
        return ""

    if output_file is None:
        output_file = f"{parse_story_url(truyen_url)[1]}.txt"

    print(f"📥 Cào {len(selected)} chương: {selected[0][0]} → {selected[-1][0]} "
          f"(song song {workers} luồng)")

    # key theo index của selected, không key theo số chương: hai mục cùng số chương
    # (vd "Chương 1" và "Chương 1: Ngoại truyện") sẽ không đè mất nội dung.
    texts: list[str | None] = [None] * len(selected)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(_throttled, download_chapter, session, num, link, delay): idx
            for idx, (num, link) in enumerate(selected)
        }
        for fut, idx in futures.items():
            _, text = fut.result()
            texts[idx] = text

    missing = [selected[idx][0] for idx, t in enumerate(texts) if t is None]
    with open(output_file, "w", encoding="utf-8") as f:
        for idx, (chap_num, _) in enumerate(selected):
            f.write(texts[idx] or f"Chương {chap_num}\n⚠️ Không có dữ liệu")
            f.write("\n\n")

    chars = sum(len(t) for t in texts if t)
    print(f"✅ Lưu {len(selected)} chương ({chars:,} ký tự) vào '{output_file}' "
          f"trong {time.perf_counter() - t0:.2f}s")
    if missing:
        print(f"⚠️ Thiếu chương: {missing}")
    return output_file


def _throttled(fn, session, chap_num, link, delay):
    time.sleep(random.uniform(0, delay))
    return fn(session, chap_num, link)


# ============================ Chạy ============================

if __name__ == "__main__":
    url = "https://toctruyen.net/truyen/avlBQwkA/ban-trai-than-tien-ho-tong"
    start_chap = 1
    end_chap = 6
    crawl_toctruyen(url, start_chap, end_chap)


📚 Truyện có 10 chương (cao nhất: chương 10)
📥 Cào 6 chương: 1 → 6 (song song 4 luồng)
✅ Lưu 6 chương (34,373 ký tự) vào 'ban-trai-than-tien-ho-tong.txt' trong 0.76s


# **Metruyen.mobi**

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

import re

def format_paragraphs(raw_text):
    # Gộp các dòng bị xuống dòng không hợp lý
    lines = raw_text.splitlines()
    paragraphs = []
    current_para = ""

    for line in lines:
        line = line.strip()
        if not line:
            # Dòng trống -> kết thúc đoạn văn
            if current_para:
                paragraphs.append(current_para.strip())
                current_para = ""
        else:
            # Nếu dòng có nội dung thì nối vào đoạn hiện tại
            if current_para:
                current_para += " " + line
            else:
                current_para = line

    if current_para:
        paragraphs.append(current_para.strip())

    # Gộp các đoạn văn lại với xuống dòng giữa mỗi đoạn
    return "\n\n".join(paragraphs)

def get_chapter_list(url):
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print("❌ Không truy cập được trang truyện.")
        return []

    # Đặt encoding đúng
    res.encoding = 'utf-8'

    soup = BeautifulSoup(res.text, "html.parser")
    chapter_tags = soup.select("li.wp-manga-chapter a")
    if not chapter_tags:
        print("❌ Không tìm thấy danh sách chương.")
        return []

    chapters = []
    for a in chapter_tags:
        title = a.text.strip()
        link = a['href']
        chapters.append((title, link))

    chapters = chapters[::-1]  # Đảo ngược để lấy từ chương 1
    for i, (title, link) in enumerate(chapters, start=1):
        print(f"{i:03d}. {title} -> {link}")
    return chapters


def get_chapter_content(title, url):
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"❌ Lỗi tải chương: {title}")
        return ""

    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, "html.parser")

    # Lấy phần nội dung
    content_div = soup.find("div", class_="text-left")
    if not content_div:
        print(f"⚠️ Không có nội dung: {title}")
        return ""

    # Xóa các thành phần bị ẩn (hiển thị:none hoặc các thẻ script, style...)
    for tag in content_div.find_all(style=lambda value: value and 'display:none' in value):
        tag.decompose()

    for tag in content_div(["script", "style"]):
        tag.decompose()

    text = content_div.get_text(separator="\n", strip=True)
    return text



def crawl_truyen_range(truyen_url, start_chap, end_chap, output_file, delay=1):
    chapters = get_chapter_list(truyen_url)
    total = len(chapters)
    print(f"📚 Tổng số chương: {total}")

    start_idx = max(start_chap - 1, 0)
    end_idx = min(end_chap, total)
    selected = chapters[start_idx:end_idx]

    with open(output_file, "w", encoding="utf-8") as f:
        for i, (title, link) in enumerate(selected, start=start_chap):
            print(f"▶️ Đang tải: [{i}] {title}")
            content = get_chapter_content(title, link)
            if content:
                formatted = format_paragraphs(content)
                f.write(f"{formatted}\n\n")
            else:
                f.write(f"{title}\n\n⚠️ Không có nội dung\n\n{'='*80}\n\n")
            time.sleep(delay)

    print(f"✅ Đã lưu chương {start_chap} → {end_chap} vào '{output_file}'")


if __name__ == "__main__":
    url = "https://metruyen.mobi/truyen/duoi-dai-tuyet-lien/"
    start_chap = 1
    end_chap = 6
    slug = url.rstrip('/').split('/')[-1]
    output_filename = f"{slug}.txt"
    crawl_truyen_range(url, start_chap, end_chap, output_file= output_filename)


📚 Truyện có 20 chương (cao nhất: chương 20)
📥 Cào 20 chương: 1 → 20 (song song 4 luồng)
✅ Lưu 20 chương (36,029 ký tự) vào 'ta-sinh-con-cho-long-ngao-thien.txt' trong 1.77s


# **monkeydtruyen**


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import re
import unicodedata
from urllib.parse import urlparse
import time

def setup_driver():
    options = Options()
    options.add_argument('--headless')  # chạy ẩn
    options.add_argument('--disable-gpu')
    options.add_argument("--no-sandbox")
    options.add_argument("window-size=1920,1080")
    options.add_argument("start-maximized")
    return webdriver.Chrome(options=options)



def clean_one_paragraph(text: str) -> str:
    # Chuẩn hoá unicode (giảm lỗi ký tự lạ)
    text = unicodedata.normalize("NFC", text)

    # Tách dòng, bỏ dòng rỗng
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    blacklist_keywords = [
        "mời quý độc giả",
        "shopee.vn",
        "truyện được đăng tải duy nhất",
        "editor:",
        "click",
        "liên kết",
        "mở ứng dụng",
        "monkeyd và đội ngũ",
        "bên dưới",
        "để tiếp tục đọc toàn bộ chương truyện",
        "https://monkeydtruyen.com/",
        "https://monkeyd.net.vn/",
    ]

    # Lọc rác theo keyword
    clean_lines = []
    for line in lines:
        lower_line = line.lower()
        if any(kw in lower_line for kw in blacklist_keywords):
            continue
        clean_lines.append(line)

    # Gộp toàn bộ thành 1 đoạn (1 chapter = 1 paragraph)
    text = " ".join(clean_lines)

    # --- Chuẩn hoá khoảng trắng ---
    text = re.sub(r"\s+", " ", text).strip()

    # --- Chuẩn hoá dấu câu: bỏ khoảng trắng trước dấu, thêm 1 khoảng trắng sau dấu (nếu cần) ---
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)               # "word ," -> "word,"
    text = re.sub(r"([,.!?;:])([^\s])", r"\1 \2", text)        # ",word" -> ", word"

    # --- Giảm dấu câu bị lặp ---
    text = re.sub(r"\.{2,}", ".", text)                        # "...." -> "."
    text = re.sub(r",{2,}", ",", text)                         # ",,," -> ","
    text = re.sub(r";{2,}", ";", text)
    text = re.sub(r":{2,}", ":", text)
    text = re.sub(r"([!?]){2,}", r"\1", text)                  # "!!!" -> "!"

    # --- Loại các ký tự rác hay gặp ---
    text = text.replace("“", "\"").replace("”", "\"").replace("’", "'").replace("…", "...")
    text = re.sub(r"\s+\)", ")", text)                         # " )" -> ")"
    text = re.sub(r"\(\s+", "(", text)                         # "( " -> "("

    # Chốt lại khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    return text


def get_chapter_list(driver, url):
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    chapter_section = soup.find("div", class_="list-chapters")
    if not chapter_section:
        print("❌ Không tìm thấy danh sách chương.")
        return []

    chapter_tags = chapter_section.find_all("div", class_="episode-title")
    chapters = []
    seen = set()

    for div in reversed(chapter_tags):  # đảo ngược để chương 1 ở đầu
        a_tag = div.find("a")
        if a_tag:
            link = a_tag['href']
            title = a_tag.get_text(strip=True)
            if link not in seen:
                seen.add(link)
                chapters.append((title, link))

    return chapters


def get_chapter_content(driver, title, url):
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # Kiểm tra có hiển thị link Shopee không
    try:
        shopee_link = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(@href, 'shopee.vn')]")))
        print(f"🔁 Chương {title} yêu cầu nhấn Shopee – đang xử lý...")

        # Click vào link Shopee
        driver.execute_script("arguments[0].click();", shopee_link)

        # Chờ chương render lại
        wait.until(EC.presence_of_element_located((By.ID, "chapter-content-render")))
        time.sleep(1)  # cho chắc chắn JS xử lý xong
    except:
        print(f"⏩ Không cần click Shopee hoặc không tìm thấy link cho chương {title}")

    # Lấy nội dung chương như bình thường
    try:
        content_div = driver.find_element(By.ID, "chapter-content-render")
        content_html = content_div.get_attribute("innerHTML")

        soup = BeautifulSoup(content_html, "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        for tag in soup.find_all(style=lambda v: v and 'display:none' in v):
            tag.decompose()

        return soup.get_text(separator="\n", strip=True)

    except Exception as e:
        print(f"❌ Lỗi khi lấy nội dung chương {title}: {e}")
        return ""

def format_paragraphs(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    blacklist_keywords = [
        "mời quý độc giả",
        "shopee.vn",
        "truyện được đăng tải duy nhất",
        "editor:",
        "click",
        "vào",
        "liên kết",
        "mở ứng dụng",
        "monkeyd và đội ngũ",
        "bên dưới",
        "để tiếp tục đọc toàn bộ chương truyện!",
        "https://monkeydtruyen.com/"
        ".]"
    ]

    clean_lines = []
    buffer = []

    for line in lines:
        lower_line = line.lower()

        # Nếu đoạn có nhiều dòng chứa keyword không mong muốn thì bỏ luôn cả cụm
        if any(kw in lower_line for kw in blacklist_keywords):
            buffer.append(line)
        else:
            if buffer:
                # Nếu có buffer tạm chứa rác, xóa nó luôn
                buffer.clear()
            clean_lines.append(line)

    return "\n\n".join(clean_lines)



def crawl_truyen_range(truyen_url, start_chap, end_chap, output_file, delay=2):
    driver = setup_driver()
    chapters = get_chapter_list(driver, truyen_url)
    total = len(chapters)
    print(f"📚 Tổng số chương: {total}")

    start_idx = max(start_chap - 1, 0)
    end_idx = min(end_chap, total)
    selected = chapters[start_idx:end_idx]

    with open(output_file, "w", encoding="utf-8") as f:
        for i, (title, link) in enumerate(selected, start=start_chap):
            print(f"▶️ Đang tải: [{i}] {title}")
            content = get_chapter_content(driver, title, link)

            if content:
                paragraph = clean_one_paragraph(content)
                if paragraph:
                    f.write(paragraph + "\n\n")   # cách nhau 1 dòng trống giữa các chương
                else:
                    f.write("\n\n")
            else:
                f.write("\n\n")

            time.sleep(delay)


    driver.quit()
    print(f"✅ Đã lưu chương {start_chap} → {end_chap} vào '{output_file}'")

def safe_filename_from_url(url: str) -> str:
    path = urlparse(url).path
    slug = path.rstrip('/').split('/')[-1]
    slug = re.sub(r'[\\/*?:"<>|]', "_", slug)
    return f"{slug}.txt"


if __name__ == "__main__":
    url = "https://monkeyd.net.vn/cong-xac.html"
    start_chap = 1
    end_chap = 10
    output_filename = safe_filename_from_url(url)

    crawl_truyen_range(url, start_chap, end_chap, output_file=output_filename)


📚 Tổng số chương: 10
▶️ Đang tải: [1] Chương 1
⏩ Không cần click Shopee hoặc không tìm thấy link cho chương Chương 1
▶️ Đang tải: [2] Chương 2
🔁 Chương Chương 2 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [3] Chương 3
🔁 Chương Chương 3 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [4] Chương 4
🔁 Chương Chương 4 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [5] Chương 5
🔁 Chương Chương 5 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [6] Chương 6
🔁 Chương Chương 6 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [7] Chương 7
🔁 Chương Chương 7 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [8] Chương 8
🔁 Chương Chương 8 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [9] Chương 9
🔁 Chương Chương 9 yêu cầu nhấn Shopee – đang xử lý...
▶️ Đang tải: [10] Chương 10 - Hết
🔁 Chương Chương 10 - Hết yêu cầu nhấn Shopee – đang xử lý...
✅ Đã lưu chương 1 → 10 vào 'cong-xac.html.txt'


# **Truyenfull.me**

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time

headers = {"User-Agent": "Mozilla/5.0"}

def get_chapter_links(truyen_url):
    res = requests.get(truyen_url, headers=headers, timeout=10)
    if res.status_code != 200:
        print(f"❌ Lỗi khi tải trang: {res.status_code}")
        return []

    soup = BeautifulSoup(res.text, "html.parser")
    chapter_tags = soup.select("div#chapters ul.chapters li a")
    if not chapter_tags:
        print("❌ Không tìm thấy danh sách chương (div#chapters → ul.chapters)")
        return []

    chapters = [(a.get("title", a.text).strip(), a["href"]) for a in chapter_tags]
    return chapters

def download_chapter(title, link, output_file):
    res = requests.get(link, headers=headers, timeout=10)
    if res.status_code != 200:
        print(f"❌ Không tải được chương: {title}")
        return
    soup = BeautifulSoup(res.text, "html.parser")
    content_div = soup.find("div", id="chapter-content")
    if not content_div:
        print(f"⚠️ Chương không chứa nội dung: {title}")
        return
    content = content_div.get_text(separator="\n").strip()
    if not content:
        print(f"⚠️ Nội dung rỗng: {title}")
        return

    output_file.write(content + "\n\n")  # Không lặp lại title chương
    print(f"✅ Đã ghi: {title}")

def crawl_truyen_range(truyen_url, start_chap, end_chap, output_file, delay=1):
    chapters = get_chapter_links(truyen_url)
    print(f"📚 Tổng chương: {len(chapters)}")
    if not chapters:
        return

    total = len(chapters)
    end_chap = end_chap or total
    start_idx = max(start_chap , 0)
    end_idx = min(end_chap, total)
    selected = chapters[start_idx:end_idx + 1]


    print(f"📥 Tải chương {start_chap} → {end_chap} ({len(selected)} chương)")
    with open(output_file, "w", encoding="utf-8") as f:
        for i, (title, link) in enumerate(selected, start=start_chap):
            print(f"▶️ [{i}] {title}")
            download_chapter(title, link, f)
            time.sleep(delay)

    print(f"\n✅ Đã lưu vào file: {output_file}")

if __name__ == "__main__":
    url = "https://truyenfull.me/story/dm-nguoi-anh-em-cau-thom-qua/391687215.html"
    start_chap = 50
    end_chap = 53
    slug = url.rstrip('/').split('/')[-2]
    output_filename = f"{slug}.txt"
    crawl_truyen_range(url, start_chap, end_chap, output_file=output_filename)



# **Truyenfull.vision**

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers = {"User-Agent": "Mozilla/5.0"}

def get_chapter_list(url):
    res = requests.get(url, headers=headers)
    res.encoding = 'utf-8'
    if res.status_code != 200:
        print("❌ Không truy cập được trang truyện.")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')
    # Selector phù hợp với cấu trúc site truyenfull.vision
    chapter_tags = soup.select('ul.list-chapter a')
    if not chapter_tags:
        print("❌ Không tìm thấy danh sách chương.")
        return []

    chapters = []
    seen = set()
    for a in reversed(chapter_tags):
        title = a.text.strip()
        link = a['href']
        if link not in seen:
            chapters.append((title, link))
            seen.add(link)

    return chapters[::-1]

def get_chapter_content(url):
    res = requests.get(url, headers=headers)
    res.encoding = 'utf-8'
    if res.status_code != 200:
        print(f"❌ Lỗi tải chương {url}")
        return ""
    soup = BeautifulSoup(res.text, 'html.parser')
    # Nội dung chính
    content_div = soup.find('div', id='chapter-c')
    if not content_div:
        print("⚠️ Không có nội dung chính.")
        return ""
    # Loại script/style và ẩn
    for tag in content_div(['script', 'style']):
        tag.decompose()
    for tag in content_div.find_all(style=lambda v: v and 'display:none' in v):
        tag.decompose()
    return content_div.get_text(separator='\n', strip=True)

def format_paragraphs(text):
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    return '\n\n'.join(lines)

def crawl_truyen_range(url, start_chap, end_chap, output_file="output.txt", delay=1):
    chapters = get_chapter_list(url)
    total = len(chapters)
    print(f"📚 Tổng chương tìm được: {total}")

    start_idx = max(start_chap-1, 0)
    end_idx = min(end_chap, total)
    selected = chapters[start_idx:end_idx]

    with open(output_file, 'w', encoding='utf-8') as f:
        for idx, (title, link) in enumerate(selected, start=start_chap):
            print(f"▶️ Chapter #{idx}: {title}")
            content = get_chapter_content(link)
            if content:
                f.write(format_paragraphs(content))
                f.write("\n\n")  # xuống 2 dòng giữa các chương
            time.sleep(delay)

    print(f"✅ Đã lưu chương {start_chap} → {end_chap} vào '{output_file}'")

if __name__ == "__main__":
    url = "https://truyenfull.vision/me-ke-o-co-dai-lam-ca-man/trang-1/#list-chapter"
    slug = url.rstrip('/').split('/')[-3]
    output_filename = f"{slug}.txt"
    crawl_truyen_range(url, start_chap=1, end_chap=5, output_file=output_filename, delay=1)


📚 Tổng chương tìm được: 50
▶️ Chapter #1: Chương 1
▶️ Chapter #2: Chương 2
▶️ Chapter #3: Chương 3
▶️ Chapter #4: Chương 4
▶️ Chapter #5: Chương 5
✅ Đã lưu chương 1 → 5 vào 'me-ke-o-co-dai-lam-ca-man.txt'


# **Tàng Thư Viện**


In [2]:
import cloudscraper
import requests
from bs4 import BeautifulSoup
import time
import re
import os

scraper = cloudscraper.create_scraper()
headers = {
    "User-Agent": "Mozilla/5.0"
}

def chinh_sua_loi_chinh_ta(text):
    replacements = {
        "“|”|\"": '"',
        "‘|’|\'": "'",
        "…": "...",
        "–|—": "-",
        "\s+": " ",
        " ,": ",",
        " \.": ".",
        "\n{2,}": "\n\n",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return text.strip()

def download_chapter(chapter_url):
    try:
        res = scraper.get(chapter_url, headers=headers, timeout=10)
        if res.status_code != 200 or "404" in res.url:
            return None

        soup = BeautifulSoup(res.text, "lxml")
        content_tag = soup.find("div", class_=lambda x: x and "box-chap" in x)

        if not content_tag:
            return None

        content = content_tag.get_text("\n", strip=True)
        content = chinh_sua_loi_chinh_ta(content)
        return content

    except Exception as e:
        print(f"❌ Lỗi khi tải {chapter_url}: {e}")
        return None
def crawl_truyen_by_loop(truyen_url, start_chap, end_chap, output_file, delay=1):
    with open(output_file, "w", encoding="utf-8") as f:
        for i in range(start_chap, end_chap + 1):
            chap_url = f"{truyen_url}/chuong-{i}"
            print(f"▶️ Đang tải: {chap_url}")
            content = download_chapter(chap_url)
            if not content:zzz
                print(f"⛔️ Dừng tại chương {i} (không tồn tại)")
                break

            f.write(f"{content}\n\n")
            print(f"✅ Đã thêm chương {i}")
            time.sleep(delay)

if __name__ == "__main__":
    url = ""
    start_chap = 1
    end_chap = 6
    slug = url.rstrip('/').split('/')[-1]
    output_filename = f"{slug}.txt"
    crawl_truyen_by_loop(url, start_chap, end_chap, output_file=output_filename)


<>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/1l/nww5s3rs3dqdvplph8xy83th0000gn/T/ipykernel_17744/1471264073.py:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  "\s+": " ",
/var/folders/1l/nww5s3rs3dqdvplph8xy83th0000gn/T/ipykernel_17744/1471264073.py:21: SyntaxWarning: "\." is an invalid escape sequence. Such s

IndentationError: unexpected indent (1471264073.py, line 54)

# **Truyendecu.net**

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

headers = {
    "User-Agent": "Mozilla/5.0"
}

 
def get_chapter_list(base_url):
    chapters = []
    seen = set()
    page = 1

    while True:
        page_url = f"{base_url}/trang-{page}" if page > 1 else base_url
        res = requests.get(page_url, headers=headers)
        res.encoding = 'utf-8'
        if res.status_code != 200:
            break

        soup = BeautifulSoup(res.text, 'html.parser')
        chapter_tags = soup.select('div.p-list-chapter a')
        if not chapter_tags:
            break

        new_chapters = 0
        for a in reversed(chapter_tags):
            title = a.text.strip()
            link = a['href']
            if link not in seen:
                chapters.append((title, link))
                seen.add(link)
                new_chapters += 1

        if new_chapters == 0:
            break

        page += 1
        time.sleep(0.5)

    return chapters[::-1]

def chapter_to_paragraph(chapter_text):

    lines = [ln.strip() for ln in chapter_text.splitlines() if ln.strip()]

    # bỏ các dòng chỉ chứa số chương, ví dụ: "01.", "1.", "001.", "Chương 1"
    filtered = []
    for ln in lines:
        if re.fullmatch(r'\d{1,3}\.', ln):  # 1., 01., 123.
            continue
        if re.fullmatch(r'0*\d{1,3}\.', ln):  # phòng cho số có 0
            continue
        if re.match(r'^(chương|chuong)\s*\d+\b', ln, flags=re.I):  # "Chương 1", "chuong 2"
            continue
        # đôi khi dòng chỉ là số đơn (ví dụ: "01" không kèm chấm)
        if re.fullmatch(r'0*\d{1,3}', ln):
            continue
        filtered.append(ln)

    # Nối tất cả thành một đoạn, cách nhau 1 khoảng trắng
    paragraph = ' '.join(filtered)
    # thu gọn nhiều khoảng trắng, xử lý khoảng trắng trước/dưới
    paragraph = re.sub(r'\s+', ' ', paragraph).strip()
    return paragraph


def get_chapter_content(url):
    res = requests.get(url, headers=headers)
    res.encoding = 'utf-8'
    if res.status_code != 200:
        return ""
    soup = BeautifulSoup(res.text, 'html.parser')
    content_div = soup.select_one('div#chapter-content')
    if not content_div:
        return ''

    # Lấy toàn bộ nội dung gốc
    lines = content_div.get_text(separator='\n').splitlines()

    # Xử lý xóa các dòng không phải nội dung truyện
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        # Bỏ qua nếu dòng rỗng, chứa 'Chương', thời gian, hoặc toàn dấu '='
        if not line:
            continue
        if line.lower().startswith('chương'):
            continue
        if 'trước' in line.lower():
            continue
        if set(line) <= {'=', '-', '*'}:
            continue
        cleaned_lines.append(line)

    return '\n'.join(cleaned_lines)


def crawl_truyen_range(url, start_chap, end_chap, output_file, delay=1):
    chapters = get_chapter_list(url)
    total = len(chapters)
    print(f"📚 Tổng chương tìm được: {total}")
    start_idx = max(start_chap - 1, 0)
    end_idx = min(end_chap, total)
    selected = chapters[start_idx:end_idx]

    with open(output_file, "w", encoding="utf-8") as f:
        for idx, (title, link) in enumerate(selected, start=start_chap):
            print(f"📥 Đang tải chương {idx}: {title}")
            content = get_chapter_content(link)
            if content:
                formatted = chapter_to_paragraph(content)
                f.write(formatted + "\n\n")
            else:
                print(f"⚠️ Không lấy được nội dung chương {idx}")
            time.sleep(delay)

    print(f"\n✅ Đã lưu {len(selected)} chương vào file '{output_file}'")


# Test crawling from a sample URL
test_url = "https://truyendecu.net/ba-noi-quy-da/"

# Lấy slug từ URL để đặt tên file
slug = test_url.rstrip('/').split('/')[-1]
output_filename = f"{slug}.txt"

crawl_truyen_range(test_url, start_chap=1, end_chap=3, output_file=output_filename)


📚 Tổng chương tìm được: 10
📥 Đang tải chương 1: Chương 1
7 ngày trước
📥 Đang tải chương 2: Chương 2
7 ngày trước
📥 Đang tải chương 3: Chương 3
7 ngày trước

✅ Đã lưu 3 chương vào file 'ba-noi-quy-da.txt'


# **Otruyen.vn**

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import re

def clean_advertising(text):
    # Xóa đoạn [Truyện được đăng tải duy nhất...]
    text = re.sub(
        r'\[Truyện được đăng tải duy nhất tại otruyen\.vn\s*-\s*https://otruyen\.vn[^\]]*\]',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    # Xóa đoạn quảng cáo Shopee
    text = re.sub(
        r'Mời Quý độc giả\s*CLICK\s*vào\s*liên kết.*?Shopee.*?cảm ơn!?',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    return text


def crawl_otruyen_with_selenium(main_url, start_chap, end_chap):
    # Lấy tên truyện từ URL
    story_slug = main_url.strip('/').split('/')[-1].replace('.html', '')
    output_file = f"{story_slug}.txt"

    # Thiết lập trình duyệt ẩn (headless)
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # Khởi tạo trình điều khiển Chrome
    driver = webdriver.Chrome(options=options)

    # Truy cập trang chính để lấy danh sách chương
    driver.get(main_url)
    time.sleep(2)  # Chờ trang tải
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Tìm danh sách chương
    chapters_div = soup.find('div', class_='list-chapters')
    if not chapters_div:
        print("❌ Không tìm thấy danh sách chương.")
        driver.quit()
        return

    chapter_links = chapters_div.find_all('a')
    chapter_links = chapter_links[::-1]  # Đảo ngược để chương 1 ở đầu

    total_chaps = len(chapter_links)
    print(f"📚 Tổng chương tìm được: {total_chaps}")

    with open(output_file, 'w', encoding='utf-8') as f:
        for i in range(start_chap - 1, min(end_chap, total_chaps)):
            chap = chapter_links[i]
            chap_title = chap.text.strip()
            chap_url = chap['href']
            print(f"▶️ Đang cào chương {i+1}: {chap_title}")

            try:
                driver.get(chap_url)
                time.sleep(2)
                chap_soup = BeautifulSoup(driver.page_source, 'html.parser')
                content_div = chap_soup.find('div', class_='chapter-content')

                if not content_div:
                    print("⚠️ Không tìm thấy nội dung chương.")
                    continue

                content_text = content_div.get_text(separator='\n', strip=True)
                content_text = clean_advertising(content_text)
                lines = content_text.split('\n')
                merged_lines = []
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue
                    if merged_lines and not merged_lines[-1].endswith(('.', '!', '?', ':', '”', '…')):
                        merged_lines[-1] += ' ' + line
                    else:
                        merged_lines.append(line)
                content_text = '\n\n'.join(merged_lines)
                f.write(f"{content_text}\n\n")

            except Exception as e:
                print(f"❌ Lỗi ở chương {i+1}: {e}")

    driver.quit()
    print(f"✅ Hoàn tất! Nội dung chương {start_chap}→{end_chap} lưu vào '{output_file}'")

# ==========================
# 🔽 Gọi hàm để sử dụng
# ==========================
main_url = "https://otruyen.vn/vo-toi-la-tin-tac.html"
crawl_otruyen_with_selenium(main_url, start_chap=1, end_chap=5)



# **Daotruyen.me**

In [6]:

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import re
import time
import os

def create_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    driver = webdriver.Chrome(options=options)
    return driver

def get_chapter_links_selenium(truyen_url):
    driver = create_driver()
    driver.get(truyen_url)
    time.sleep(5)  # Chờ trang load xong

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    # Lấy tên truyện từ <title>
    title_tag = soup.find("title")
    truyen_title = title_tag.text.strip().split(" - ")[0] if title_tag else "unknown"

    # Tìm bảng danh sách chương
    table = soup.select_one("div.lg\\:mx-16 table.w-full")
    if not table:
        print("❌ Không tìm thấy bảng danh sách chương.")
        return [], truyen_title

    base_url = "https://daotruyen.me"
    chapter_tags = table.select("tbody td a")

    chapters = []
    for a in chapter_tags:
        text = a.text.strip()
        match = re.search(r"Chương\s+(\d+)", text)
        if match:
            chap_num = int(match.group(1))
            full_url = base_url + a["href"]
            chapters.append((chap_num, full_url))

    chapters.sort(key=lambda x: x[0])

    # ✅ In ra toàn bộ danh sách chương
    print(f"\n📄 Danh sách chương ({len(chapters)}):")
    for chap_num, link in chapters:
        print(f" - Chương {chap_num}: {link}")

    return chapters, truyen_title

# Hàm lấy nội dung chương, xử lý popup quảng cáo
def get_chapter_content(driver, title, url):
    driver.get(url)
    wait = WebDriverWait(driver, 30)

    # Popup xử lý như cũ
    popups = driver.find_elements(By.XPATH, "//div[contains(@class,'relative') and contains(@class,'p-6')]")
    if popups:
        print(f"🔁 {title} yêu cầu mở khóa – đang click link quảng cáo...")
        ad_link = popups[0].find_element(By.CSS_SELECTOR, "a.text-blue-500")
        ad_link.click()
        time.sleep(5)
    else:
        print(f"⏩ Không cần mở khóa chương {title}")

    # Đợi container có text thật
    try:
        content_container = wait.until(
            EC.presence_of_element_located((By.CLASS_NAME, "read_str_content"))
        )
        wait.until(lambda d: content_container.text.strip() != "")
    except:
        print(f"⚠️ Không tìm thấy nội dung cho {title}")
        return ""

    # Lấy innerText thay vì innerHTML
    content_text = content_container.get_attribute("innerText")

    lines = content_text.splitlines()
    cleaned_lines = []
    for line in lines:
        l = line.strip().lower()
        if re.fullmatch(r"\d{1,4}", l):
            continue
        if l in ["hết", "hết chương", "hoàn", "hoàn thành", "(hoàn)"]:
            continue
        if line.strip():
            cleaned_lines.append(line.strip())

    if cleaned_lines:
        return "\n".join(cleaned_lines)
    else:
        print(f"⚠️ Nội dung rỗng ở {title}")
        return ""



# Hàm crawl theo khoảng chương
def crawl_truyen_range(truyen_url, start_chap, end_chap, output_file, delay=1):
    chapters, truyen_title = get_chapter_links_selenium(truyen_url)
    if not chapters:
        print("❌ Không có chương nào.")
        return

    start_chap = max(start_chap, 1)
    filtered = [(num, link) for num, link in chapters if start_chap <= num <= end_chap]
    if not filtered:
        print("❌ Không tìm thấy chương trong khoảng yêu cầu.")
        return

    print(f"\n📥 Tải chương {start_chap} → {end_chap} ({len(filtered)} chương)")

    driver = create_driver()

    with open(output_file, "w", encoding="utf-8") as f:
        for chap_num, link in filtered:
            title = f"Chương {chap_num}"
            print(f"\n▶️ {title}")
            content = get_chapter_content(driver, title, link)
            if content:
                f.write(f"{content}\n\n")
                print(f"✅ Đã lưu {title}")
            time.sleep(delay)

    driver.quit()
    print(f"\n✅ Đã lưu toàn bộ vào: {output_file}")

# Chạy chính
if __name__ == "__main__":
    url = "https://daotruyen.me/mieu-nu-am-ta"
    start_chap = 1
    end_chap = 6

    # Dùng lại get_chapter_links_selenium để lấy tên truyện
    _, truyen_title = get_chapter_links_selenium(url)
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", truyen_title)
    output_filename = f"{safe_name}.txt"

    crawl_truyen_range(url, start_chap, end_chap, output_file=output_filename)



📄 Danh sách chương (6):
 - Chương 1: https://daotruyen.me/mieu-nu-am-ta/chuong-1.html
 - Chương 2: https://daotruyen.me/mieu-nu-am-ta/chuong-2.html
 - Chương 3: https://daotruyen.me/mieu-nu-am-ta/chuong-3.html
 - Chương 4: https://daotruyen.me/mieu-nu-am-ta/chuong-4.html
 - Chương 5: https://daotruyen.me/mieu-nu-am-ta/chuong-5.html
 - Chương 6: https://daotruyen.me/mieu-nu-am-ta/chuong-6.html

📄 Danh sách chương (6):
 - Chương 1: https://daotruyen.me/mieu-nu-am-ta/chuong-1.html
 - Chương 2: https://daotruyen.me/mieu-nu-am-ta/chuong-2.html
 - Chương 3: https://daotruyen.me/mieu-nu-am-ta/chuong-3.html
 - Chương 4: https://daotruyen.me/mieu-nu-am-ta/chuong-4.html
 - Chương 5: https://daotruyen.me/mieu-nu-am-ta/chuong-5.html
 - Chương 6: https://daotruyen.me/mieu-nu-am-ta/chuong-6.html

📥 Tải chương 1 → 6 (6 chương)

▶️ Chương 1
⏩ Không cần mở khóa chương Chương 1
✅ Đã lưu Chương 1

▶️ Chương 2
🔁 Chương 2 yêu cầu mở khóa – đang click link quảng cáo...
✅ Đã lưu Chương 2

▶️ Chương 3
⏩ Khôn

# **Truyenmacothat.net**

In [11]:
import time
import re
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from concurrent.futures import ThreadPoolExecutor, as_completed


# ======== Lấy danh sách chương + tên truyện bằng Selenium (chỉ chạy 1 lần) ========
def get_chapter_links_selenium(url):
    print("🔍 Đang tải trang truyện bằng Selenium...")
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(3)

    # Nhấn “Show more” nếu có
    try:
        while True:
            show_more = driver.find_element("css selector", ".chapter-readmore")
            if show_more.is_displayed():
                driver.execute_script("arguments[0].click();", show_more)
                time.sleep(0.5)
            else:
                break
    except Exception:
        pass

    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")
    title_tag = soup.select_one("div.post-title")
    truyen_title = title_tag.get_text(strip=True) if title_tag else "TruyenKhongTen"

    chapter_elements = soup.select("li.wp-manga-chapter a")
    chapters = [(a.text.strip(), a["href"]) for a in chapter_elements]
    chapters.reverse()

    print(f"✅ Đã lấy được {len(chapters)} chương của truyện: {truyen_title}")
    return chapters, truyen_title


# ======== Cào nội dung 1 chương ========
def crawl_chapter(title, url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        content_div = soup.select_one("div.reading-content")
        if not content_div:
            print(f"⚠️ Không thấy nội dung {title}")
            return ""
        paragraphs = content_div.find_all("p")
        text = "\n".join(p.get_text(strip=True) for p in paragraphs).strip()
        return text
    except Exception as e:
        print(f"❌ Lỗi khi cào {title}: {e}")
        return ""


# ======== Cào trong khoảng chương, dùng đa luồng ========
def crawl_chapter_range(url, start_chap, end_chap, output_file):
    chapters, truyen_title = get_chapter_links_selenium(url)
    total_chaps = len(chapters)
    start_chap = max(1, start_chap)
    end_chap = min(total_chaps, end_chap)

    selected_chaps = chapters[start_chap - 1:end_chap]
    print(f"📖 Đang cào {len(selected_chaps)} chương ({start_chap} → {end_chap}) song song...")

    all_texts = [""] * len(selected_chaps)

    with ThreadPoolExecutor(max_workers=8) as executor:  # 8 luồng để tải nhanh hơn
        futures = {
            executor.submit(crawl_chapter, title, link): idx
            for idx, (title, link) in enumerate(selected_chaps)
        }

        for future in as_completed(futures):
            idx = futures[future]
            try:
                result = future.result()
                all_texts[idx] = result
            except Exception as e:
                print(f"⚠️ Lỗi ở chương {idx + start_chap}: {e}")

    full_text = "\n\n".join(all_texts)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_text)

    print(f"\n🎉 Hoàn tất! Truyện đã lưu vào file: {output_file}")


# ======== MAIN ========
if __name__ == "__main__":
    url = "https://truyenmacothat.net/truyen/thanh-giao/"
    start_chap = 1
    end_chap = 6

    # Dùng selenium để lấy tên truyện
    _, truyen_title = get_chapter_links_selenium(url)
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", truyen_title)
    output_filename = f"{safe_name}.txt"

    crawl_chapter_range(url, start_chap, end_chap, output_file=output_filename)


🔍 Đang tải trang truyện bằng Selenium...
✅ Đã lấy được 8 chương của truyện: THÁNH GIÁO
🔍 Đang tải trang truyện bằng Selenium...
✅ Đã lấy được 8 chương của truyện: THÁNH GIÁO
📖 Đang cào 6 chương (1 → 6) song song...

🎉 Hoàn tất! Truyện đã lưu vào file: THÁNH GIÁO.txt


# **Xóa Thư Mục**

In [ ]:
import os
import glob

folder_path = "/content/truyen_391687215"
file_list = glob.glob(os.path.join(folder_path, "*.txt"))

for file_path in file_list:
    try:
        os.remove(file_path)
        print(f"Đã xóa: {file_path}")
    except OSError as e:
        print(f"Lỗi khi xóa {file_path}: {e}")

print(f"Hoàn thành việc xóa file .txt trong thư mục {folder_path}.")

Đã xóa: /content/truyen_391687215/Chương 84 Hậu kì Viên mãn.txt
Đã xóa: /content/truyen_391687215/Chương 83 Khai pháo đầu năm.txt
Đã xóa: /content/truyen_391687215/Chương 99 Món quà ngoại truyện.txt
Đã xóa: /content/truyen_391687215/Chương 88 PN Thislife 3.txt
Đã xóa: /content/truyen_391687215/Chương 96 PN Cùng sống lại.txt
Đã xóa: /content/truyen_391687215/Chương 90 PN Trúc mã 2.txt
Đã xóa: /content/truyen_391687215/Chương 51 Bọn mình cũng đi thuê khách sạn đi.txt
Đã xóa: /content/truyen_391687215/Chương 85.1 PN Trình Mặc Phỉ hồi phục ký ức.txt
Đã xóa: /content/truyen_391687215/Chương 1 Tân sinh viên nhập học.txt
Đã xóa: /content/truyen_391687215/Chương 87 PN Thislife 2.txt
Đã xóa: /content/truyen_391687215/Chương 2 Thơm quá.txt
Đã xóa: /content/truyen_391687215/Chương 86 PN Thislife.txt
Đã xóa: /content/truyen_391687215/Chương 89 PN Trúc Mã.txt
Đã xóa: /content/truyen_391687215/Chương 49 Ai bảo giữa gay và trai thẳng không có tình bạn thuần khiết.txt
Đã xóa: /content/truyen_391687215